In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import requests

# Load and preprocess data
df = pd.read_csv('/Users/gautam/Downloads/Training.csv', parse_dates=['Completion Date', 'Match Activation Date', 'Big Birthdate', 'Little Birthdate'])

# Calculate Match Length
df['Match Length'] = (df['Completion Date'] - df['Match Activation Date']).dt.days

# Handle missing values
df.fillna({
    'Closure Reason': 'Active',
    'Closure Details': 'N/A',
    'Match Support Contact Notes': '',
    'Rationale for Match': ''
}, inplace=True)

# EDA: Closure Reason Distribution by Program Type
plt.figure(figsize=(12,6))
sns.countplot(data=df, x='Program Type', hue='Closure Reason')
plt.xticks(rotation=45)
plt.title('Closure Reason Distribution by Program Type')
plt.show()

# Temporal Analysis: Matches Over Time
df['Year'] = df['Completion Date'].dt.year
yearly_matches = df.groupby('Year').size()
yearly_matches.plot(kind='bar', title='Matches Per Year')

# Text Analysis: TF-IDF and Sentiment
tfidf = TfidfVectorizer(max_features=50, stop_words='english')
notes_tfidf = tfidf.fit_transform(df['Match Support Contact Notes'])
keywords = pd.DataFrame(notes_tfidf.toarray(), columns=tfidf.get_feature_names_out())

# Sentiment Analysis
df['Sentiment'] = df['Match Support Contact Notes'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# Predictive Modeling Features
X = pd.concat([
    df[['Program Type', 'Big Age', 'Little Age', 'Sentiment']],
    keywords,
    pd.get_dummies(df[['Big Gender', 'Little Gender']])
], axis=1)
y = df['Match Length']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'Model RMSE: {rmse:.2f} days')

# Census Data Integration (Example)
def get_census_data(block_group):
    response = requests.get(
        f'https://api.census.gov/data/2020/acs/acs5?get=B01001_001E,B19013_001E&for=block%20group:{block_group}&in=state:27&key=YOUR_API_KEY'
    )
    return response.json()

# Example usage
census_data = get_census_data('270531109003')
print('Census Data:', census_data)

# Intervention Strategy Questions
intervention_questions = [
    "Has there been any recent change in communication frequency?",
    "Are there upcoming major life events for either Big or Little?",
    "Have you noticed changes in Little's engagement level?",
    "Are there unresolved logistical challenges (transportation, scheduling)?"
]
print("\nRecommended Intervention Questions:")
for q in intervention_questions:
    print(f"- {q}")

ModuleNotFoundError: No module named 'textblob'